# Load the glucose readings and count the "Out of range" readings

**Objective.** Read the Lingo export into R and find out what the text "Out of range" in the glucose column stands for, so that no reading is dropped by mistake in later notebooks.

This notebook follows the literate programming approach of Knuth (1984, *Computer Journal* 27:97-111): each short piece of code is followed by a plain explanation. It also follows Rule et al. (2019, *PLoS Computational Biology* 15:e1007007), Ten simple rules for writing and sharing computational analyses in Jupyter Notebooks.

The data are in `../data/` and the environment is `../glucose.yml`.

## Step 1. Read the file

In [1]:
glucose <- read.csv("../data/lingo-glucose-2026-09-27-to-2026-11-17.csv",
                    stringsAsFactors = FALSE)
head(glucose)

  time_local_with_offset glucose_mg_dl
1 2026-09-27T08:17-05:00           138
2 2026-09-27T08:22-05:00           131
3 2026-09-27T08:27-05:00           127
4 2026-09-27T08:32-05:00           142
5 2026-09-27T08:37-05:00           135
6 2026-09-27T08:42-05:00           131


**Explanation**

* `read.csv("../data/...csv", ...)` reads a comma separated file. The `../` means one folder up from `ipynb`, then into `data`.
* `stringsAsFactors = FALSE` keeps text as text. R then does not turn the glucose column into categories, which matters because that column holds both numbers and the words "Out of range".
* `glucose <- ...` stores the table under the name `glucose`. The name is ours.
* `head(glucose)` shows the first 6 rows.

In [2]:
nrow(glucose)
range(glucose$time_local_with_offset)

[1] 13558
[1] "2026-09-27T08:17-05:00" "2026-11-17T16:08-05:00"


**Explanation**

* `nrow(glucose)` counts the rows. Each row is one reading.
* `range(...)` gives the smallest and largest value of the time column. Because the times are written as year-month-day-hour-minute, the smallest is also the earliest.

**Result.** The file has 13,558 readings, from 2026-09-27 08:17 to 2026-11-17 16:08.

## Step 2. Find the readings that are not numbers

In [3]:
glucose$out_of_range <- glucose$glucose_mg_dl == "Out of range"
glucose$glucose <- suppressWarnings(as.numeric(glucose$glucose_mg_dl))
table(out_of_range = glucose$out_of_range)
sum(is.na(glucose$glucose) & !glucose$out_of_range)

out_of_range
FALSE  TRUE 
13438   120 
[1] 0


**Explanation**

* `glucose$out_of_range <- glucose$glucose_mg_dl == "Out of range"` adds a column that is TRUE where the text is exactly "Out of range" and FALSE everywhere else. The `==` asks whether two things are equal.
* `as.numeric(...)` turns text into numbers. Where the text is not a number, it gives `NA`, which is R's mark for a missing value. `suppressWarnings(...)` hides the warning R prints about this, since we expect it here.
* `table(out_of_range = ...)` counts how many rows are TRUE and how many are FALSE. The name before `=` is only a label for the printout.
* The last line counts rows that are missing a number but are not marked "Out of range". For example, a blank or a misspelled word would be counted. If it prints 0, then "Out of range" is the only text in the column.

**Result.** 120 readings are "Out of range". No other text appears in the glucose column.

## Step 3. See what values sit next to them

In [4]:
glucose$before <- c(NA, head(glucose$glucose, -1))
glucose$after  <- c(tail(glucose$glucose, -1), NA)
neighbours <- glucose[glucose$out_of_range, c("before", "after")]
summary(neighbours)

     before          after      
 Min.   :55.00   Min.   :55.00  
 1st Qu.:55.00   1st Qu.:55.00  
 Median :56.00   Median :56.00  
 Mean   :56.24   Mean   :58.64  
 3rd Qu.:57.00   3rd Qu.:58.00  
 Max.   :61.00   Max.   :87.00  
 NA's   :95      NA's   :95     


**Explanation**

* `head(glucose$glucose, -1)` is the glucose column without its last value. Putting an `NA` in front of it moves every value down one row, so each row now holds the reading from 5 minutes earlier. This is the `before` column.
* `tail(glucose$glucose, -1)` is the column without its first value. Putting an `NA` at the end moves every value up one row, so each row holds the reading from 5 minutes later. This is the `after` column.
* `glucose[glucose$out_of_range, c("before", "after")]` keeps only the rows marked "Out of range", and only these two columns.
* `summary(...)` gives the smallest, middle and largest values. It leaves out rows where the neighbour is itself missing, and reports how many it left out as `NA's`. Several "Out of range" readings come in a row, so many neighbours are also "Out of range".

**Result.** The 120 readings come in 25 separate stretches. Where the neighbouring reading is a number, it is between 55 and 61 on the earlier side, and between 55 and 87 on the later side, with a middle value of 56. The readings next to "Out of range" are at the low end of everything the sensor reported. In this data the smallest number the sensor reports is 55.

In this export "Out of range" therefore appears to mean a value below what the sensor reports, at or under 54. We have not yet checked this against the sensor's user guide. Later notebooks count these readings as low, and say so each time.

## Step 4. Check the time zone on every row

In [5]:
table(offset = substr(glucose$time_local_with_offset, 17, 22))

offset
-05:00 
 13558 


**Explanation**

* `substr(x, 17, 22)` takes characters 17 to 22 of each time. In `2026-09-27T08:17-05:00` these are the last six characters, `-05:00`, which is the time zone offset from Greenwich time.
* `table(...)` counts how many rows have each offset.

**Result.** All 13,558 rows have the offset `-05:00`. An offset of `-05:00` fits United States Central daylight time. Central clocks moved back one hour on 2026-11-01, which would change the offset to `-06:00`. The offset never changes in this file, so the times after 2026-11-01 may not follow the wall clock. In notebook "time_of_day" we check this against the daily rhythm before using times of day.